# EP02. Re-ranking 적용기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있습니다:

1. **Two-Stage Retrieval** 아키텍처의 원리와 필요성 이해
2. LangChain `ContextualCompressionRetriever` + `CohereRerank` 구현
3. `MRR@5`, `Recall@5` 지표로 1-Stage vs 2-Stage 정확도 정량 비교
4. `top_k` 파라미터에 따른 지연시간/정확도 트레이드오프 실험
5. Langfuse로 Reranking 파이프라인 추적 및 모니터링

---

## AI 가이드 안내

> **이 노트북에서 AI를 활용하는 법**
>
> - 각 섹션 상단의 `# TODO` 주석을 AI에게 맡기거나 직접 구현해 보세요.
> - 막히는 부분은 해당 셀을 복사해 Claude에게 "이 코드가 작동하지 않아요. 오류: ..." 형태로 물어보세요.
> - Exercise는 힌트를 먼저 읽고 스스로 시도한 뒤 AI의 리뷰를 받는 방식을 권장합니다.

**예상 소요 시간:** 약 60~90분  
**필요 API 키:** `ANTHROPIC_API_KEY`, `COHERE_API_KEY` (없으면 BGE 로컬 fallback 사용), `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`

---

## 섹션 1. 환경 설정

필요한 패키지를 설치합니다. `uv`를 사용하면 빠르게 설치할 수 있습니다.

In [ ]:
# uv pip install을 사용한 패키지 설치
import subprocess, sys

packages = [
    "langchain",
    "langchain-anthropic",
    "langchain-community",
    "langchain-cohere",
    "langchain-chroma",
    "langfuse",
    "chromadb",
    "sentence-transformers",
    "rank-bm25",
    "python-dotenv",
    "matplotlib",
    "pandas",
]

# uv가 설치된 경우 uv pip install 사용, 아니면 pip 사용
try:
    subprocess.run(["uv", "--version"], check=True, capture_output=True)
    cmd = ["uv", "pip", "install"] + packages
    print("uv를 사용하여 패키지를 설치합니다...")
except (subprocess.CalledProcessError, FileNotFoundError):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
    print("pip을 사용하여 패키지를 설치합니다...")

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("설치 완료!")
else:
    print("설치 오류:\n", result.stderr[:500])

---

## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from dotenv import load_dotenv

from langchain.schema import Document
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# .env 파일에서 API 키 로드
load_dotenv()

# API 키 확인
keys = {
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    "COHERE_API_KEY": os.getenv("COHERE_API_KEY"),
    "LANGFUSE_PUBLIC_KEY": os.getenv("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.getenv("LANGFUSE_SECRET_KEY"),
}

for k, v in keys.items():
    status = "설정됨" if v else "미설정 (일부 기능 제한)"
    print(f"{k}: {status}")

USE_COHERE = bool(os.getenv("COHERE_API_KEY"))
USE_LANGFUSE = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))
print(f"\nCohere Rerank 사용: {USE_COHERE}")
print(f"Langfuse 추적 사용: {USE_LANGFUSE}")

---

## 섹션 3. 한국어 기술 문서 및 테스트 쿼리 정의

실험에 사용할 한국어 기술 문서 15개와 테스트 쿼리 5개를 인라인으로 정의합니다.

In [ ]:
# 한국어 기술 문서 15개 정의
DOCUMENTS_RAW = [
    {
        "id": "doc_01",
        "content": """분산 트랜잭션이란 여러 노드에 걸쳐 원자적으로 실행되어야 하는 트랜잭션입니다.
데이터베이스 클러스터 환경에서 ACID 속성을 보장하기 위해 사용됩니다.
대표적인 구현 방법으로는 2PC(Two-Phase Commit), Saga 패턴 등이 있습니다.""",
        "metadata": {"topic": "distributed_transaction", "level": "intro"}
    },
    {
        "id": "doc_02",
        "content": """2PC(Two-Phase Commit) 프로토콜은 분산 트랜잭션의 원자성을 보장하는 알고리즘입니다.
Prepare 단계와 Commit 단계로 나뉩니다.
코디네이터가 모든 참여자에게 준비 완료 여부를 확인한 후 최종 커밋을 결정합니다.""",
        "metadata": {"topic": "2pc", "level": "concept"}
    },
    {
        "id": "doc_03",
        "content": """2PC의 핵심 단점은 코디네이터 장애 시 블로킹(Blocking) 문제입니다.
코디네이터가 Prepare 단계 후 장애가 발생하면 모든 참여자가 대기 상태에 빠집니다.
이로 인해 타임아웃이 발생하고 전체 트랜잭션이 롤백되거나 무기한 대기할 수 있습니다.
단일 장애점(SPOF) 문제로 고가용성 시스템에서는 사용을 지양합니다.""",
        "metadata": {"topic": "2pc", "level": "deep", "subtopic": "limitations"}
    },
    {
        "id": "doc_04",
        "content": """Saga 패턴은 분산 트랜잭션을 여러 개의 로컬 트랜잭션으로 분해합니다.
각 로컬 트랜잭션 실패 시 보상 트랜잭션(Compensating Transaction)을 실행합니다.
코레오그래피(Choreography)와 오케스트레이션(Orchestration) 두 가지 구현 방식이 있습니다.
2PC와 달리 블로킹 문제가 없어 마이크로서비스 환경에 적합합니다.""",
        "metadata": {"topic": "saga", "level": "concept"}
    },
    {
        "id": "doc_05",
        "content": """Kubernetes는 컨테이너화된 애플리케이션의 자동 배포, 스케일링, 관리를 위한 오픈소스 플랫폼입니다.
Pod, Service, Deployment, ConfigMap 등의 리소스 타입을 제공합니다.
Google이 내부적으로 사용하던 Borg 시스템을 기반으로 2014년 오픈소스로 공개했습니다.""",
        "metadata": {"topic": "kubernetes", "level": "intro"}
    },
    {
        "id": "doc_06",
        "content": """Kubernetes HPA(Horizontal Pod Autoscaler)는 CPU, 메모리, 커스텀 메트릭을 기반으로
자동으로 Pod 수를 조절합니다. metrics-server가 설치되어야 동작하며,
스케일 업/다운 쿨다운 기간을 설정해 과도한 스케일링을 방지합니다.""",
        "metadata": {"topic": "kubernetes", "level": "deep", "subtopic": "hpa"}
    },
    {
        "id": "doc_07",
        "content": """RAG(Retrieval-Augmented Generation)는 언어 모델에 외부 지식을 주입하는 기법입니다.
벡터 데이터베이스에서 관련 문서를 검색하여 LLM 컨텍스트로 제공합니다.
환각(Hallucination)을 줄이고 최신 정보를 반영할 수 있습니다.""",
        "metadata": {"topic": "rag", "level": "intro"}
    },
    {
        "id": "doc_08",
        "content": """벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 유사도 검색을 지원합니다.
대표적으로 Chroma, Pinecone, Weaviate, Milvus가 있습니다.
ANN(Approximate Nearest Neighbor) 알고리즘으로 빠른 검색을 지원합니다.
HNSW, IVF, PQ 등의 인덱싱 알고리즘이 사용됩니다.""",
        "metadata": {"topic": "vector_db", "level": "concept"}
    },
    {
        "id": "doc_09",
        "content": """임베딩 모델은 텍스트를 고차원 벡터로 변환합니다.
OpenAI text-embedding-3-small, Cohere embed-multilingual-v3, BGE-m3 등이 있습니다.
한국어 지원 모델로는 jhgan/ko-sroberta-multitask, snunlp/KR-ELECTRA-discriminator 등이 있습니다.""",
        "metadata": {"topic": "embedding", "level": "concept"}
    },
    {
        "id": "doc_10",
        "content": """Redis는 인메모리 데이터 구조 저장소로 캐시, 세션 저장, 메시지 브로커로 사용됩니다.
String, Hash, List, Set, Sorted Set, Stream 등의 데이터 타입을 지원합니다.
Pub/Sub 기능과 TTL(Time-To-Live) 설정으로 캐시 만료를 자동 처리합니다.""",
        "metadata": {"topic": "redis", "level": "concept"}
    },
    {
        "id": "doc_11",
        "content": """GraphQL은 REST API의 over-fetching, under-fetching 문제를 해결하는 쿼리 언어입니다.
클라이언트가 필요한 데이터의 구조를 직접 정의하여 요청합니다.
단일 엔드포인트에서 쿼리, 뮤테이션, 서브스크립션을 모두 처리합니다.""",
        "metadata": {"topic": "graphql", "level": "concept"}
    },
    {
        "id": "doc_12",
        "content": """Apache Kafka는 분산 이벤트 스트리밍 플랫폼입니다.
토픽(Topic), 파티션(Partition), 오프셋(Offset) 개념을 기반으로 합니다.
높은 처리량과 내구성을 보장하며 실시간 데이터 파이프라인 구축에 사용됩니다.
소비자 그룹(Consumer Group)으로 병렬 처리를 지원합니다.""",
        "metadata": {"topic": "kafka", "level": "concept"}
    },
    {
        "id": "doc_13",
        "content": """마이크로서비스 아키텍처는 애플리케이션을 독립적으로 배포 가능한 서비스로 분리합니다.
각 서비스는 단일 책임 원칙(SRP)을 따르며 자체 데이터베이스를 가집니다.
서비스 간 통신은 REST, gRPC, 메시지 큐를 통해 이루어집니다.""",
        "metadata": {"topic": "microservices", "level": "concept"}
    },
    {
        "id": "doc_14",
        "content": """CI/CD 파이프라인은 코드 변경 사항을 자동으로 빌드, 테스트, 배포하는 프로세스입니다.
GitHub Actions, Jenkins, GitLab CI 등의 도구를 사용합니다.
블루-그린 배포, 카나리 배포 등의 전략으로 무중단 배포를 구현합니다.""",
        "metadata": {"topic": "cicd", "level": "concept"}
    },
    {
        "id": "doc_15",
        "content": """서비스 메시(Service Mesh)는 마이크로서비스 간 네트워크 통신을 관리하는 인프라 계층입니다.
Istio, Linkerd가 대표적인 구현체입니다.
트래픽 관리, 서비스 검색, 로드 밸런싱, 보안(mTLS), 관찰성을 제공합니다.
사이드카 패턴(Envoy 프록시)으로 애플리케이션 코드 변경 없이 기능을 추가합니다.""",
        "metadata": {"topic": "service_mesh", "level": "concept"}
    },
]

# 테스트 쿼리 5개 + 정답 문서 ID
TEST_QUERIES = [
    {
        "query": "분산 트랜잭션에서 2PC 프로토콜의 단점과 한계는 무엇인가요?",
        "relevant_ids": ["doc_03", "doc_02"],  # doc_03이 가장 관련성 높음
        "primary_id": "doc_03"
    },
    {
        "query": "마이크로서비스에서 분산 트랜잭션을 처리하는 블로킹 없는 방법",
        "relevant_ids": ["doc_04", "doc_13", "doc_01"],
        "primary_id": "doc_04"
    },
    {
        "query": "Kubernetes에서 Pod을 자동으로 스케일링하는 방법",
        "relevant_ids": ["doc_06", "doc_05"],
        "primary_id": "doc_06"
    },
    {
        "query": "RAG에서 벡터 검색 시 사용되는 임베딩 모델 종류",
        "relevant_ids": ["doc_09", "doc_08", "doc_07"],
        "primary_id": "doc_09"
    },
    {
        "query": "서비스 간 네트워크 통신을 애플리케이션 수정 없이 관리하는 방법",
        "relevant_ids": ["doc_15", "doc_13"],
        "primary_id": "doc_15"
    },
]

# LangChain Document 객체로 변환
documents = [
    Document(
        page_content=d["content"],
        metadata={**d["metadata"], "doc_id": d["id"]}
    )
    for d in DOCUMENTS_RAW
]

print(f"문서 수: {len(documents)}")
print(f"테스트 쿼리 수: {len(TEST_QUERIES)}")
print("\n샘플 문서:")
print(documents[0].page_content[:100], "...")

---

## 섹션 4. 1-Stage 검색 구현 (ChromaDB + HuggingFaceEmbeddings)

In [ ]:
print("임베딩 모델 로드 중... (최초 실행 시 모델 다운로드 필요)")

# 다국어 임베딩 모델 (한국어 지원)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("ChromaDB 벡터스토어 구축 중...")

# ChromaDB 인메모리 인스턴스 생성
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="ep02_reranking_demo"
)

print(f"벡터스토어 구축 완료: {vectorstore._collection.count()}개 문서")

# 1-Stage Base Retriever 생성
base_retriever_k5 = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# 동작 확인
test_results = base_retriever_k5.invoke(TEST_QUERIES[0]["query"])
print(f"\n[1-Stage] 쿼리: '{TEST_QUERIES[0]['query'][:40]}...'")
print("검색 결과 (상위 5개):")
for i, doc in enumerate(test_results, 1):
    doc_id = doc.metadata.get('doc_id', 'unknown')
    is_relevant = doc_id in TEST_QUERIES[0]['relevant_ids']
    marker = "[관련]" if is_relevant else "[비관련]"
    print(f"  {i}. {marker} {doc_id}: {doc.page_content[:60]}...")

---

## 섹션 5. CohereRerank 통합 (ContextualCompressionRetriever)

**[Two-Stage Retrieval 아키텍처]**
```mermaid
flowchart LR
    Q((사용자<br/>쿼리)):::query --> R(1단계: Retriever<br/>Bi-Encoder<br/>상위 50~100개 후보):::stage1
    R --> RR(2단계: Reranker<br/>Cross-Encoder<br/>상위 5~10개 정제):::stage2
    RR --> LLM([✨ LLM 답변 생성]):::llm
    
    classDef query fill:#607d8b,stroke:#fff,color:#fff,stroke-width:2px,font-weight:bold
    classDef stage1 fill:#2196f3,stroke:#fff,color:#fff,stroke-width:2px,rx:10px,ry:10px
    classDef stage2 fill:#ff9800,stroke:#fff,color:#fff,stroke-width:2px,rx:10px,ry:10px
    classDef llm fill:#4caf50,stroke:#fff,color:#fff,stroke-width:2px,rx:5px,ry:5px
```


In [ ]:
from langchain.retrievers import ContextualCompressionRetriever

# Cohere Rerank를 사용한 2-Stage Retriever
if USE_COHERE:
    from langchain_cohere import CohereRerank
    
    # base_retriever: 넉넉히 20개 후보 검색
    base_retriever_for_rerank = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 20}
    )
    
    cohere_reranker = CohereRerank(
        model="rerank-multilingual-v3.0",
        top_n=5
    )
    
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=cohere_reranker,
        base_retriever=base_retriever_for_rerank
    )
    
    print("[Cohere Rerank] Two-Stage Retriever 초기화 완료")
    print("  - Base Retriever: k=20 (후보 20개)")
    print("  - Cohere Rerank: rerank-multilingual-v3.0, top_n=5")
    
    # 동작 확인
    test_results_reranked = compression_retriever.invoke(TEST_QUERIES[0]["query"])
    print(f"\n[2-Stage] 쿼리: '{TEST_QUERIES[0]['query'][:40]}...'")
    print("Reranking 후 결과 (상위 5개):")
    for i, doc in enumerate(test_results_reranked, 1):
        doc_id = doc.metadata.get('doc_id', 'unknown')
        is_relevant = doc_id in TEST_QUERIES[0]['relevant_ids']
        relevance_score = doc.metadata.get('relevance_score', 'N/A')
        marker = "[관련]" if is_relevant else "[비관련]"
        print(f"  {i}. {marker} {doc_id} (점수: {relevance_score:.4f} if {relevance_score} != 'N/A' else 'N/A'): {doc.page_content[:50]}...")
else:
    print("[주의] COHERE_API_KEY가 설정되지 않았습니다.")
    print("섹션 6에서 BGE Reranker로 자동 전환됩니다.")

---

## 섹션 6. BGE Cross-Encoder Fallback (COHERE_API_KEY 없을 때)

In [ ]:
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

def create_reranker(top_n: int = 5, base_k: int = 20):
    """Cohere API 키 유무에 따라 적절한 Reranker 반환"""
    base_ret = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": base_k}
    )
    
    if USE_COHERE:
        from langchain_cohere import CohereRerank
        compressor = CohereRerank(
            model="rerank-multilingual-v3.0",
            top_n=top_n
        )
        reranker_name = "Cohere Rerank (multilingual-v3.0)"
    else:
        # BGE Reranker v2 (다국어 지원)
        cross_encoder = HuggingFaceCrossEncoder(
            model_name="BAAI/bge-reranker-v2-m3"
        )
        compressor = CrossEncoderReranker(
            model=cross_encoder,
            top_n=top_n
        )
        reranker_name = "BGE Reranker v2-m3 (로컬)"
    
    retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=base_ret
    )
    
    return retriever, reranker_name


# 기본 설정으로 Reranker 생성
print("Reranker 초기화 중... (BGE 로컬 모델은 최초 실행 시 다운로드 필요)")
reranker, reranker_name = create_reranker(top_n=5, base_k=20)
print(f"사용 중인 Reranker: {reranker_name}")

# 전체 파이프라인에서 사용할 retriever로 설정
compression_retriever = reranker

---

## 섹션 7. MRR@5, Recall@5 측정 함수

In [ ]:
def calculate_mrr(retrieved_ids: list, relevant_ids: list, k: int = 5) -> float:
    """
    MRR@k (Mean Reciprocal Rank) 계산
    retrieved_ids: 검색된 문서 ID 목록 (순서대로)
    relevant_ids: 관련 문서 ID 목록
    """
    for rank, doc_id in enumerate(retrieved_ids[:k], 1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def calculate_recall(retrieved_ids: list, relevant_ids: list, k: int = 5) -> float:
    """
    Recall@k 계산
    """
    if not relevant_ids:
        return 0.0
    retrieved_top_k = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)
    return len(retrieved_top_k & relevant_set) / len(relevant_set)


def calculate_precision_at_1(retrieved_ids: list, relevant_ids: list) -> float:
    """Precision@1 계산"""
    if not retrieved_ids:
        return 0.0
    return 1.0 if retrieved_ids[0] in relevant_ids else 0.0


def evaluate_retriever(retriever, test_queries: list, k: int = 5) -> dict:
    """
    Retriever 성능 평가
    Returns: {mrr, recall, p@1, latency_ms, per_query_results}
    """
    mrr_scores = []
    recall_scores = []
    p1_scores = []
    latencies = []
    per_query = []
    
    for q_data in test_queries:
        query = q_data["query"]
        relevant_ids = q_data["relevant_ids"]
        
        start = time.perf_counter()
        results = retriever.invoke(query)
        latency_ms = (time.perf_counter() - start) * 1000
        
        retrieved_ids = [doc.metadata.get("doc_id", "") for doc in results]
        
        mrr = calculate_mrr(retrieved_ids, relevant_ids, k)
        recall = calculate_recall(retrieved_ids, relevant_ids, k)
        p1 = calculate_precision_at_1(retrieved_ids, relevant_ids)
        
        mrr_scores.append(mrr)
        recall_scores.append(recall)
        p1_scores.append(p1)
        latencies.append(latency_ms)
        
        per_query.append({
            "query": query[:50],
            "retrieved_ids": retrieved_ids,
            "relevant_ids": relevant_ids,
            "mrr": mrr,
            "recall": recall,
            "p@1": p1,
            "latency_ms": latency_ms
        })
    
    return {
        "mrr": sum(mrr_scores) / len(mrr_scores),
        "recall": sum(recall_scores) / len(recall_scores),
        "p@1": sum(p1_scores) / len(p1_scores),
        "avg_latency_ms": sum(latencies) / len(latencies),
        "per_query": per_query
    }


# 측정 함수 단위 테스트
assert calculate_mrr(["doc_03", "doc_02"], ["doc_03"], k=5) == 1.0
assert calculate_mrr(["doc_01", "doc_03"], ["doc_03"], k=5) == 0.5
assert calculate_mrr(["doc_01", "doc_02"], ["doc_03"], k=5) == 0.0
assert calculate_recall(["doc_03", "doc_01"], ["doc_03", "doc_02"], k=5) == 0.5
print("측정 함수 단위 테스트 통과!")

---

## 섹션 8. 1-Stage vs 2-Stage 정확도 비교 실험

**[Bi-Encoder vs Cross-Encoder 비교]**
```mermaid
flowchart TD
    subgraph BE ["⚡ Bi-Encoder (빠른 1차 검색)"]
        direction TB
        Q1[/"사용자 쿼리"\]:::node --> EQ("🤖 쿼리 인코더"):::enc
        D1[/"문서 코퍼스"\]:::node --> ED("🤖 문서 인코더"):::enc
        EQ --> SIM{"부분 유사도<br/>(미리 계산 가능)"}:::sim
        ED --> SIM
    end
    
    subgraph CE ["🎯 Cross-Encoder (정밀한 2차 재정렬)"]
        direction TB
        Q2[/"사용자 쿼리"\]:::node --> CAT("➕ [쿼리 + 문서]<br/>텍스트 연결 후 연산"):::cat
        D2[/"문서 후보군"\]:::node --> CAT
        CAT --> SCORE{"딥러닝 문맥 분석<br/>(실시간 높은 연산)"}:::score
    end

    classDef node fill:#eceff1,stroke:#b0bec5,color:#37474f,stroke-width:2px
    classDef enc fill:#1e88e5,stroke:#fff,color:#fff,stroke-width:2px
    classDef cat fill:#8e24aa,stroke:#fff,color:#fff,stroke-width:2px
    classDef sim fill:#fb8c00,stroke:#fff,color:#fff,stroke-width:2px,font-weight:bold
    classDef score fill:#e53935,stroke:#fff,color:#fff,stroke-width:2px,font-weight:bold
```


In [ ]:
print("=" * 60)
print("1-Stage vs 2-Stage 정확도 비교 실험")
print("=" * 60)

# 1-Stage 평가
print("\n[1단계] 1-Stage Retriever 평가 중...")
stage1_results = evaluate_retriever(base_retriever_k5, TEST_QUERIES, k=5)
print(f"  MRR@5:    {stage1_results['mrr']:.4f}")
print(f"  Recall@5: {stage1_results['recall']:.4f}")
print(f"  P@1:      {stage1_results['p@1']:.4f}")
print(f"  평균 지연: {stage1_results['avg_latency_ms']:.1f}ms")

# 2-Stage 평가
print("\n[2단계] 2-Stage Retriever 평가 중...")
stage2_results = evaluate_retriever(compression_retriever, TEST_QUERIES, k=5)
print(f"  MRR@5:    {stage2_results['mrr']:.4f}")
print(f"  Recall@5: {stage2_results['recall']:.4f}")
print(f"  P@1:      {stage2_results['p@1']:.4f}")
print(f"  평균 지연: {stage2_results['avg_latency_ms']:.1f}ms")

# 개선율 계산
mrr_improvement = (stage2_results['mrr'] - stage1_results['mrr']) / max(stage1_results['mrr'], 1e-9) * 100
latency_increase = (stage2_results['avg_latency_ms'] - stage1_results['avg_latency_ms']) / max(stage1_results['avg_latency_ms'], 1e-9) * 100

print("\n" + "-" * 40)
print("개선 결과:")
print(f"  MRR@5 개선:   +{mrr_improvement:.1f}%")
print(f"  지연시간 증가: +{latency_increase:.1f}%")

# 결과 데이터프레임
comparison_df = pd.DataFrame([
    {"방법": "1-Stage (Bi-Encoder)", "MRR@5": stage1_results['mrr'],
     "Recall@5": stage1_results['recall'], "P@1": stage1_results['p@1'],
     "평균 지연(ms)": stage1_results['avg_latency_ms']},
    {"방법": f"2-Stage ({reranker_name})", "MRR@5": stage2_results['mrr'],
     "Recall@5": stage2_results['recall'], "P@1": stage2_results['p@1'],
     "평균 지연(ms)": stage2_results['avg_latency_ms']}
])

print("\n결과 요약 표:")
print(comparison_df.to_string(index=False, float_format="{:.4f}".format))

---

## 섹션 9. Latency 측정 (time.perf_counter)

In [ ]:
import statistics

def measure_latency(retriever, query: str, n_runs: int = 5) -> dict:
    """동일 쿼리를 n_runs번 실행하여 지연시간 통계 측정"""
    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        retriever.invoke(query)
        latencies.append((time.perf_counter() - start) * 1000)
    
    return {
        "mean_ms": statistics.mean(latencies),
        "median_ms": statistics.median(latencies),
        "stdev_ms": statistics.stdev(latencies) if len(latencies) > 1 else 0,
        "min_ms": min(latencies),
        "max_ms": max(latencies),
    }

test_query = TEST_QUERIES[0]["query"]
print(f"지연시간 측정 쿼리: '{test_query[:50]}...'")
print(f"반복 횟수: 5회\n")

print("1-Stage 지연시간 측정 중...")
s1_latency = measure_latency(base_retriever_k5, test_query, n_runs=5)
print(f"  평균: {s1_latency['mean_ms']:.1f}ms | 중간값: {s1_latency['median_ms']:.1f}ms | "
      f"표준편차: {s1_latency['stdev_ms']:.1f}ms")

print("2-Stage 지연시간 측정 중...")
s2_latency = measure_latency(compression_retriever, test_query, n_runs=5)
print(f"  평균: {s2_latency['mean_ms']:.1f}ms | 중간값: {s2_latency['median_ms']:.1f}ms | "
      f"표준편차: {s2_latency['stdev_ms']:.1f}ms")

overhead_ms = s2_latency['mean_ms'] - s1_latency['mean_ms']
print(f"\nReranking 오버헤드: +{overhead_ms:.1f}ms")

---

## 섹션 10. top_k 파라미터 튜닝 실험 (k=3, 5, 10, 20)

In [ ]:
print("top_k 파라미터 튜닝 실험")
print("각 k값에 대해 2-Stage Retriever를 생성하고 성능 측정")
print("=" * 60)

k_values = [5, 10, 20, 50]
tuning_results = []

for base_k in k_values:
    print(f"\nbase_k={base_k} 실험 중...")
    
    # 각 k값으로 새로운 retriever 생성
    ret, rname = create_reranker(top_n=5, base_k=base_k)
    
    # 성능 평가
    result = evaluate_retriever(ret, TEST_QUERIES, k=5)
    
    tuning_results.append({
        "base_k": base_k,
        "mrr@5": result["mrr"],
        "recall@5": result["recall"],
        "latency_ms": result["avg_latency_ms"]
    })
    
    print(f"  MRR@5={result['mrr']:.4f}, "
          f"Recall@5={result['recall']:.4f}, "
          f"지연={result['avg_latency_ms']:.1f}ms")

# 효율성 지수 계산
tuning_df = pd.DataFrame(tuning_results)
baseline_mrr = tuning_df.iloc[0]['mrr@5']
baseline_latency = tuning_df.iloc[0]['latency_ms']

tuning_df['mrr_gain_%'] = ((tuning_df['mrr@5'] - baseline_mrr) / max(baseline_mrr, 1e-9) * 100).round(1)
tuning_df['latency_increase_%'] = ((tuning_df['latency_ms'] - baseline_latency) / max(baseline_latency, 1e-9) * 100).round(1)
tuning_df['efficiency_index'] = (
    tuning_df['mrr_gain_%'] / tuning_df['latency_increase_%'].replace(0, 1)
).round(3)

print("\n\n튜닝 결과 테이블:")
print(tuning_df.to_string(index=False, float_format="{:.4f}".format))

best_k = tuning_df.loc[tuning_df['efficiency_index'].idxmax(), 'base_k']
print(f"\n최적 base_k (효율성 지수 기준): {int(best_k)}")

---

## 섹션 11. Langfuse CallbackHandler 통합

In [ ]:
if USE_LANGFUSE:
    from langfuse.langchain import CallbackHandler
    
    langfuse_handler = CallbackHandler(
        public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    )
    print("Langfuse CallbackHandler 초기화 완료")
    
    # LLM 파이프라인 구성
    if os.getenv("ANTHROPIC_API_KEY"):
        llm = ChatAnthropic(
            model="claude-3-5-haiku-20241022",
            max_tokens=512
        )
        
        prompt = ChatPromptTemplate.from_template(
            """다음 컨텍스트를 참고하여 질문에 답하세요.
            
컨텍스트:
{context}

질문: {question}

답변:"""
        )
        
        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)
        
        # Two-Stage RAG 체인 구성
        rag_chain = (
            {
                "context": compression_retriever | format_docs,
                "question": RunnablePassthrough()
            }
            | prompt
            | llm
            | StrOutputParser()
        )
        
        # Langfuse 추적과 함께 실행
        test_query = TEST_QUERIES[0]["query"]
        print(f"\nLangfuse 추적 테스트 실행...")
        print(f"쿼리: '{test_query}'")
        
        answer = rag_chain.invoke(
            test_query,
            config={"callbacks": [langfuse_handler]}
        )
        
        print(f"\n답변:\n{answer}")
        print("\nLangfuse 대시보드에서 추적 결과를 확인하세요: https://cloud.langfuse.com")
    else:
        print("[주의] ANTHROPIC_API_KEY가 없어 LLM 체인 테스트를 건너뜁니다.")
else:
    print("[주의] LANGFUSE_PUBLIC_KEY / SECRET_KEY가 설정되지 않았습니다.")
    print("Langfuse 추적 없이 진행합니다.")
    langfuse_handler = None

---

## 섹션 12. 결과 시각화

In [ ]:
# 한글 폰트 설정 (macOS)
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Linux':
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 그래프 1: 정확도 비교 막대 그래프 ---
ax1 = axes[0]
metrics = ['MRR@5', 'Recall@5', 'P@1']
stage1_vals = [stage1_results['mrr'], stage1_results['recall'], stage1_results['p@1']]
stage2_vals = [stage2_results['mrr'], stage2_results['recall'], stage2_results['p@1']]

x = range(len(metrics))
width = 0.35
bars1 = ax1.bar([i - width/2 for i in x], stage1_vals, width,
                label='1-Stage (Bi-Encoder)', color='#93c5fd', edgecolor='white')
bars2 = ax1.bar([i + width/2 for i in x], stage2_vals, width,
                label=f'2-Stage ({reranker_name[:15]})', color='#34d399', edgecolor='white')

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax1.set_xlabel('지표')
ax1.set_ylabel('점수')
ax1.set_title('1-Stage vs 2-Stage 정확도 비교')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.set_ylim(0, 1.15)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# --- 그래프 2: top_k 별 MRR@5 vs Latency scatter ---
ax2 = axes[1]
latencies = tuning_df['latency_ms'].tolist()
mrrs = tuning_df['mrr@5'].tolist()
k_vals = tuning_df['base_k'].tolist()

scatter = ax2.scatter(latencies, mrrs, s=200, c=k_vals,
                      cmap='viridis', zorder=5, edgecolors='white', linewidth=2)

for i, (lat, mrr, k) in enumerate(zip(latencies, mrrs, k_vals)):
    ax2.annotate(f'k={int(k)}', (lat, mrr),
                 textcoords="offset points", xytext=(8, 5),
                 fontsize=10, fontweight='bold')

ax2.plot(latencies, mrrs, '--', color='gray', alpha=0.5, zorder=4)
plt.colorbar(scatter, ax=ax2, label='base_k 값')
ax2.set_xlabel('평균 지연시간 (ms)')
ax2.set_ylabel('MRR@5')
ax2.set_title('top_k별 Latency vs 정확도 트레이드오프')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ep02_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장 완료: ep02_results.png")

---

## Exercise

### Exercise 1: Reranking 전후 MRR@5 비교

**목표:** 추가 쿼리 5개를 직접 정의하고 1-Stage vs 2-Stage MRR@5를 비교하세요.

**힌트:**
- 문서 목록을 참고하여 정답이 명확한 쿼리를 설계하세요.
- `evaluate_retriever()` 함수를 재활용하세요.
- 두 방식의 MRR@5 차이와 개선율을 출력하세요.

In [ ]:
# Exercise 1: 나만의 테스트 쿼리 5개 정의
MY_TEST_QUERIES = [
    # TODO: 아래 형식으로 5개 쿼리를 정의하세요
    {
        "query": "여기에 쿼리를 작성하세요",
        "relevant_ids": ["doc_XX"],  # 관련 문서 ID
        "primary_id": "doc_XX"       # 가장 관련성 높은 문서
    },
    # ... 4개 더 추가
]

# TODO: 1-Stage와 2-Stage 평가 실행 및 결과 비교
# my_stage1 = evaluate_retriever(base_retriever_k5, MY_TEST_QUERIES)
# my_stage2 = evaluate_retriever(compression_retriever, MY_TEST_QUERIES)
# ...

### Exercise 2: top_k 트레이드오프 실험 확장

**목표:** `top_n` 파라미터(최종 반환 문서 수)도 변수로 추가하여 `base_k × top_n` 조합 실험을 진행하세요.

**힌트:**
- `base_k = [10, 20, 50]`, `top_n = [3, 5, 10]` 조합 (9가지 실험)
- 각 조합의 MRR@5와 지연시간을 측정하세요.
- 히트맵(seaborn `heatmap`)으로 결과를 시각화하면 좋습니다.
- 최적 (base_k, top_n) 조합을 찾고 근거를 서술하세요.

In [ ]:
# Exercise 2: base_k × top_n 조합 실험
# TODO: 아래 코드를 완성하세요

base_k_list = [10, 20, 50]
top_n_list = [3, 5, 10]

# results_grid = {}
# for base_k in base_k_list:
#     for top_n in top_n_list:
#         ret, _ = create_reranker(top_n=top_n, base_k=base_k)
#         result = evaluate_retriever(ret, TEST_QUERIES, k=top_n)
#         results_grid[(base_k, top_n)] = result
#         print(f"base_k={base_k}, top_n={top_n}: MRR={result['mrr']:.4f}, Latency={result['avg_latency_ms']:.1f}ms")

# 보너스: 결과를 히트맵으로 시각화
# import seaborn as sns
# ...